# Refine

> Postprocessing markdown files including fixing heading hierarchy and adding image descriptions

In [ ]:
#| default_exp refine

This module aims to fix and enrich markdown headings from OCR'd PDF files by:

1. Fixing heading hierarchy that was corrupted during OCR
2. Adding page numbers to headings for better navigation
3. Enriching figure references with descriptive text and creating a table of figures

In [ ]:
#| export
from fastcore.all import *
from mistocr.core import read_pgs
from re import sub, findall, MULTILINE
from pydantic import BaseModel
from lisette.core import completion
import os
import json

# Dev only
import sys
sys.path.append('../../..')
from ctx_utils import *

# Lisette Core

# AUTOGENERATED! DO NOT EDIT! File to edit: ../nbs/00_core.ipynb.

# %% auto 0
__all__ = ['sonn45', 'detls_tag', 're_tools', 'effort', 'patch_litellm', 'remove_cache_ckpts', 'mk_msg', 'fmt2hist', 'mk_msgs',
           'stream_with_complete', 'lite_mk_func', 'ToolResponse', 'cite_footnote', 'cite_footnotes', 'Chat',
           'random_tool_id', 'mk_tc', 'mk_tc_req', 'mk_tc_result', 'mk_tc_results', 'astream_with_complete',
           'AsyncChat', 'mk_tr_details', 'AsyncStreamFormatter', 'adisplay_stream']

# %% ../nbs/00_core.ipynb
import asyncio, base64, json, litellm, mimetypes, random, string
from typing import Optional
from html import escape
from litellm import (acompletion, completion, stream_chunk_builder, Message,
                     ModelResponse, ModelResponseStream, get_model_info, register_model, Usage)
from litellm.utils import function_to_dict, StreamingChoices, Delta, ChatCompletionMessageToolCall, Function, Choices
from toolslm.funccall import mk_ns, call_func, call_func_async, get_schema
from fastcore.utils import *
from fastcore import imghdr
from dataclasses import dataclass

# %% ../nbs/00_core.ipynb
def patch_litellm(seed=0):
    "Patch litellm.ModelResponseBase such that `id` and `created` are fixed."
    from litellm.types.utils import ModelResponseBase
    @patch
    def __init__(self: ModelResponseBase, id=None, created=None, *args, **kwargs): 
        self._orig___init__(id='chatcmpl-xxx', created=1000000000, *args, **kwargs)

    @patch
    def __setattr__(self: ModelResponseBase, name, value):
        if name == 'id': value = 'chatcmpl-xxx'
        elif name == 'created': value = 1000000000
        self._orig___setattr__(name, value)

    if seed is not None: random.seed(seed) # ensures random ids like tool call ids are deterministic

# %% ../nbs/00_core.ipynb
@patch
def _repr_markdown_(self: litellm.ModelResponse):
    message = self.choices[0].message
    content = ''
    if mc:=message.content: content += mc[0]['text'] if isinstance(mc,list) else mc
    if message.tool_calls:
        tool_calls = [f"\n\n🔧 {nested_idx(tc,'function','name')}({nested_idx(tc,'function','arguments')})\n" for tc in message.tool_calls]
        content += "\n".join(tool_calls)
    if not content: content = str(message)
    details = [
        f"id: `{self.id}`",
        f"model: `{self.model}`",
        f"finish_reason: `{self.choices[0].finish_reason}`"
    ]
    if hasattr(self, 'usage') and self.usage: details.append(f"usage: `{self.usage}`")
    det_str = '\n- '.join(details)
    
    return f"""{content}

<details>

- {det_str}

</details>"""

# %% ../nbs/00_core.ipynb
register_model({
    "claude-sonnet-4-5": {
        "max_tokens": 64000, "max_input_tokens": 200000, "max_output_tokens": 64000,
        "input_cost_per_token": 3e-06, "output_cost_per_token": 1.5e-05, "cache_creation_input_token_cost": 3.75e-06, "cache_read_input_token_cost": 3e-07,
        "litellm_provider": "anthropic", "mode": "chat",
        "supports_function_calling": True, "supports_parallel_function_calling": True, "supports_vision": True, "supports_prompt_caching": True, "supports_response_schema": True, "supports_system_messages": True, "supports_reasoning": True, "supports_assistant_prefill": True,
        "supports_tool_choice": True, "supports_computer_use": True 
    }
});
sonn45 = "claude-sonnet-4-5"

# %% ../nbs/00_core.ipynb
def _bytes2content(data):
    "Convert bytes to litellm content dict (image or pdf)"
    mtype = 'application/pdf' if data[:4] == b'%PDF' else mimetypes.types_map.get(f'.{imghdr.what(None, h=data)}')
    if not mtype: raise ValueError(f'Data must be image or PDF bytes, got {data[:10]}')
    return {'type': 'image_url', 'image_url': f'data:{mtype};base64,{base64.b64encode(data).decode("utf-8")}'}

# %% ../nbs/00_core.ipynb
def _add_cache_control(msg,          # LiteLLM formatted msg
                       ttl=None):    # Cache TTL: '5m' (default) or '1h'
    "cache `msg` with default time-to-live (ttl) of 5minutes ('5m'), but can be set to '1h'."
    if isinstance(msg["content"], str): 
        msg["content"] = [{"type": "text", "text": msg["content"]}]
    cache_control = {"type": "ephemeral"}
    if ttl is not None: cache_control["ttl"] = ttl
    if isinstance(msg["content"], list) and msg["content"]:
        msg["content"][-1]["cache_control"] = cache_control
    return msg

def _has_cache(msg):
    return msg["content"] and isinstance(msg["content"], list) and ('cache_control' in msg["content"][-1])

def remove_cache_ckpts(msg):
    "remove cache checkpoints and return msg."
    if _has_cache(msg): msg["content"][-1].pop('cache_control', None)
    return msg

def _mk_content(o):
    if isinstance(o, str): return {'type':'text','text':o.strip() or '.'}
    elif isinstance(o,bytes): return _bytes2content(o)
    return o

# %% ../nbs/00_core.ipynb
def mk_msg(
    content,      # Content: str, bytes (image), list of mixed content, or dict w 'role' and 'content' fields
    role="user",  # Message role if content isn't already a dict/Message
    cache=False,  # Enable Anthropic caching
    ttl=None      # Cache TTL: '5m' (default) or '1h'
):
    "Create a LiteLLM compatible message."
    if isinstance(content, dict) or isinstance(content, Message): return content
    if isinstance(content, ModelResponse): return content.choices[0].message
    if isinstance(content, list) and len(content) == 1 and isinstance(content[0], str): c = content[0]
    elif isinstance(content, list): c = [_mk_content(o) for o in content]
    else: c = content
    msg = {"role": role, "content": c}
    return _add_cache_control(msg, ttl=ttl) if cache else msg

# %% ../nbs/00_core.ipynb
detls_tag = "<details class='tool-usage-details'>"
re_tools = re.compile(fr"^({detls_tag}\n+```json\n+(.*?)\n+```\n+</details>)", flags=re.DOTALL|re.MULTILINE)

# %% ../nbs/00_core.ipynb
def _extract_tool(text:str)->tuple[dict,dict]:
    "Extract tool call and results from <details> block"
    d = json.loads(text.strip())
    call = d['call']
    func = call['function']
    tc = ChatCompletionMessageToolCall(Function(dumps(call['arguments']),func), d['id'])
    tr = {'role': 'tool','tool_call_id': d['id'],'name': func, 'content': d['result']}
    return tc,tr

def fmt2hist(outp:str)->list:
    "Transform a formatted output into a LiteLLM compatible history"
    lm,hist = Message(),[]
    spt = re_tools.split(outp)
    for txt,_,tooljson in chunked(spt, 3, pad=True):
        txt = txt.strip() if tooljson or txt.strip() else '.'
        hist.append(lm:=Message(txt))
        if tooljson:
            tcr = _extract_tool(tooljson)
            if not hist: hist.append(lm) # if LLM calls a tool without talking
            lm.tool_calls = lm.tool_calls+[tcr[0]] if lm.tool_calls else [tcr[0]] 
            hist.append(tcr[1])
    return hist

# %% ../nbs/00_core.ipynb
def _apply_cache_idxs(msgs, cache_idxs=[-1], ttl=None):
    'Add cache control to idxs after filtering tools'
    ms = L(msgs).filter(lambda m: not (m.get('tool_calls', []) or m['role'] == 'tool'))
    for i in cache_idxs:
        try: _add_cache_control(ms[i], ttl)
        except IndexError: continue

# %% ../nbs/00_core.ipynb
def mk_msgs(
    msgs,                   # List of messages (each: str, bytes, list, or dict w 'role' and 'content' fields)
    cache=False,            # Enable Anthropic caching
    cache_idxs=[-1],        # Cache breakpoint idxs
    ttl=None,               # Cache TTL: '5m' (default) or '1h'
):
    "Create a list of LiteLLM compatible messages."
    if not msgs: return []
    if not isinstance(msgs, list): msgs = [msgs]
    res,role = [],'user'
    msgs = L(msgs).map(lambda m: fmt2hist(m) if detls_tag in m else [m]).concat()
    for m in msgs:
        res.append(msg:=remove_cache_ckpts(mk_msg(m, role=role)))
        role = 'assistant' if msg['role'] in ('user','function', 'tool') else 'user'
    if cache: _apply_cache_idxs(res, cache_idxs, ttl)
    return res

# %% ../nbs/00_core.ipynb
def stream_with_complete(gen, postproc=noop):
    "Extend streaming response chunks with the complete response"
    chunks = []
    for chunk in gen:
        chunks.append(chunk)
        yield chunk
    postproc(chunks)
    return stream_chunk_builder(chunks)

# %% ../nbs/00_core.ipynb
def lite_mk_func(f):
    if isinstance(f, dict): return f
    return {'type':'function', 'function':get_schema(f, pname='parameters')}

# %% ../nbs/00_core.ipynb
@dataclass
class ToolResponse:
    content: list[str,str]

# %% ../nbs/00_core.ipynb
def _lite_call_func(tc,ns,raise_on_err=True):
    try: fargs = json.loads(tc.function.arguments)
    except Exception as e: raise ValueError(f"Failed to parse function arguments: {tc.function.arguments}") from e
    res = call_func(tc.function.name, fargs,ns=ns)
    if isinstance(res, ToolResponse): res = res.content
    else: res = str(res)
    return {"tool_call_id": tc.id, "role": "tool", "name": tc.function.name, "content": res}

# %% ../nbs/00_core.ipynb
def _has_search(m):
    i = get_model_info(m)
    return bool(i.get('search_context_cost_per_query') or i.get('supports_web_search'))

# %% ../nbs/00_core.ipynb
def cite_footnote(msg):
    if not (delta:=nested_idx(msg, 'choices', 0, 'delta')): return
    if citation:= nested_idx(delta, 'provider_specific_fields', 'citation'):
        title = citation['title'].replace('"', '\\"')
        delta.content = f'[*]({citation["url"]} "{title}") '
        
def cite_footnotes(stream_list):
    "Add markdown footnote citations to stream deltas"
    for msg in stream_list: cite_footnote(msg)

# %% ../nbs/00_core.ipynb
effort = AttrDict({o[0]:o for o in ('low','medium','high')})

# %% ../nbs/00_core.ipynb
def _mk_prefill(pf): return ModelResponseStream([StreamingChoices(delta=Delta(content=pf,role='assistant'))])

# %% ../nbs/00_core.ipynb
_final_prompt = "You have no more tool uses. Please summarize your findings. If you did not complete your goal please tell the user what further work needs to be done so they can choose how best to proceed."

# %% ../nbs/00_core.ipynb
class Chat:
    def __init__(
        self,
        model:str,                # LiteLLM compatible model name 
        sp='',                    # System prompt
        temp=0,                   # Temperature
        search=False,             # Search (l,m,h), if model supports it
        tools:list=None,          # Add tools
        hist:list=None,           # Chat history
        ns:Optional[dict]=None,   # Custom namespace for tool calling 
        cache=False,              # Anthropic prompt caching
        cache_idxs:list=[-1],     # Anthropic cache breakpoint idxs, use `0` for sys prompt if provided
        ttl=None,                 # Anthropic prompt caching ttl
    ):
        "LiteLLM chat client."
        self.model = model
        hist,tools = mk_msgs(hist,cache,cache_idxs,ttl),listify(tools)
        if ns is None and tools: ns = mk_ns(tools)
        elif ns is None: ns = globals()
        self.tool_schemas = [lite_mk_func(t) for t in tools] if tools else None
        store_attr()
    
    def _prep_msg(self, msg=None, prefill=None):
        "Prepare the messages list for the API call"
        sp = [{"role": "system", "content": self.sp}] if self.sp else []
        if sp:
            if 0 in self.cache_idxs: sp[0] = _add_cache_control(sp[0])
            cache_idxs = L(self.cache_idxs).filter().map(lambda o: o-1 if o>0 else o)
        else:
            cache_idxs = self.cache_idxs
        if msg: self.hist = mk_msgs(self.hist+[msg], self.cache, cache_idxs, self.ttl)
        pf = [{"role":"assistant","content":prefill}] if prefill else []
        return sp + self.hist + pf

    def _call(self, msg=None, prefill=None, temp=None, think=None, search=None, stream=False, max_steps=2, step=1, final_prompt=None, tool_choice=None, **kwargs):
        "Internal method that always yields responses"
        if step>max_steps: return
        if not get_model_info(self.model).get("supports_assistant_prefill"): prefill=None
        if _has_search(self.model) and (s:=ifnone(search,self.search)): kwargs['web_search_options'] = {"search_context_size": effort[s]}
        else: _=kwargs.pop('web_search_options',None)
        res = completion(model=self.model, messages=self._prep_msg(msg, prefill), stream=stream, 
                         tools=self.tool_schemas, reasoning_effort = effort.get(think), tool_choice=tool_choice,
                         # temperature is not supported when reasoning
                         temperature=None if think else ifnone(temp,self.temp),
                         **kwargs)
        if stream:
            if prefill: yield _mk_prefill(prefill)
            res = yield from stream_with_complete(res,postproc=cite_footnotes)
        m = res.choices[0].message
        if prefill: m.content = prefill + m.content
        self.hist.append(m)
        yield res

        if tcs := m.tool_calls:
            tool_results=[_lite_call_func(tc, ns=self.ns) for tc in tcs]
            self.hist+=tool_results
            for r in tool_results: yield r
            if step>=max_steps-1: prompt,tool_choice,search = final_prompt,'none',False
            else: prompt = None
            yield from self._call(
                prompt, prefill, temp, think, search, stream, max_steps, step+1,
                final_prompt, tool_choice, **kwargs)
    
    def __call__(self,
                 msg=None,          # Message str, or list of multiple message parts
                 prefill=None,      # Prefill AI response if model supports it
                 temp=None,         # Override temp set on chat initialization
                 think=None,        # Thinking (l,m,h)
                 search=None,       # Override search set on chat initialization (l,m,h)
                 stream=False,      # Stream results
                 max_steps=2, # Maximum number of tool calls
                 final_prompt=_final_prompt, # Final prompt when tool calls have ran out 
                 return_all=False,  # Returns all intermediate ModelResponses if not streaming and has tool calls
                 **kwargs):
        "Main call method - handles streaming vs non-streaming"
        result_gen = self._call(msg, prefill, temp, think, search, stream, max_steps, 1, final_prompt, **kwargs)     
        if stream: return result_gen              # streaming
        elif return_all: return list(result_gen)  # toolloop behavior
        else: return last(result_gen)             # normal chat behavior

# %% ../nbs/00_core.ipynb
@patch
def print_hist(self:Chat):
    "Print each message on a different line"
    for r in self.hist: print(r, end='\n\n')

# %% ../nbs/00_core.ipynb
def random_tool_id():
    "Generate a random tool ID with 'toolu_' prefix"
    random_part = ''.join(random.choices(string.ascii_letters + string.digits, k=25))
    return f'toolu_{random_part}'

# %% ../nbs/00_core.ipynb
def mk_tc(func, args, tcid=None, idx=1):
    if not tcid: tcid = random_tool_id()
    return {'index': idx, 'function': {'arguments': args, 'name': func}, 'id': tcid, 'type': 'function'}

# %% ../nbs/00_core.ipynb
def mk_tc_req(content, tcs):
    msg = Message(content=content, role='assistant', tool_calls=tcs, function_call=None)
    msg.tool_calls = [{**dict(tc), 'function': dict(tc['function'])} for tc in msg.tool_calls]
    return msg

# %% ../nbs/00_core.ipynb
def mk_tc_result(tc, result): return {'tool_call_id': tc['id'], 'role': 'tool', 'name': tc['function']['name'], 'content': result}

# %% ../nbs/00_core.ipynb
def mk_tc_results(tcq, results): return [mk_tc_result(a,b) for a,b in zip(tcq.tool_calls, results)]

# %% ../nbs/00_core.ipynb
async def _alite_call_func(tc, ns, raise_on_err=True):
    try: fargs = json.loads(tc.function.arguments)
    except Exception as e: raise ValueError(f"Failed to parse function arguments: {tc.function.arguments}") from e
    res = await call_func_async(tc.function.name, fargs, ns=ns)
    if isinstance(res, ToolResponse): res = res.content
    else: res = str(res)
    return {"tool_call_id": tc.id, "role": "tool", "name": tc.function.name, "content": res}

# %% ../nbs/00_core.ipynb
@asave_iter
async def astream_with_complete(self, agen, postproc=noop):
    chunks = []
    async for chunk in agen:
        chunks.append(chunk)
        postproc(chunk)
        yield chunk
    self.value = stream_chunk_builder(chunks)

# %% ../nbs/00_core.ipynb
class AsyncChat(Chat):
    async def _call(self, msg=None, prefill=None, temp=None, think=None, search=None, stream=False, max_steps=2, step=1, final_prompt=None, tool_choice=None, **kwargs):
        if step>max_steps+1: return
        if not get_model_info(self.model).get("supports_assistant_prefill"): prefill=None
        if _has_search(self.model) and (s:=ifnone(search,self.search)): kwargs['web_search_options'] = {"search_context_size": effort[s]}
        else: _=kwargs.pop('web_search_options',None)
        res = await acompletion(model=self.model, messages=self._prep_msg(msg, prefill), stream=stream,
                         tools=self.tool_schemas, reasoning_effort=effort.get(think), tool_choice=tool_choice,
                         # temperature is not supported when reasoning
                         temperature=None if think else ifnone(temp,self.temp), 
                         **kwargs)
        if stream:
            if prefill: yield _mk_prefill(prefill)
            res = astream_with_complete(res,postproc=cite_footnote)
            async for chunk in res: yield chunk
            res = res.value
        m=res.choices[0].message
        if prefill: m.content = prefill + m.content
        yield res
        self.hist.append(m)

        if tcs := m.tool_calls:
            tool_results = []
            for tc in tcs:
                result = await _alite_call_func(tc, ns=self.ns)
                tool_results.append(result)
                yield result
            self.hist+=tool_results
            if step>=max_steps-1: prompt,tool_choice,search = final_prompt,'none',False
            else: prompt = None
            async for result in self._call(
                prompt, prefill, temp, think, search, stream, max_steps, step+1,
                final_prompt, tool_choice=tool_choice, **kwargs):
                    yield result
    
    async def __call__(self,
                       msg=None,          # Message str, or list of multiple message parts
                       prefill=None,      # Prefill AI response if model supports it
                       temp=None,         # Override temp set on chat initialization
                       think=None,        # Thinking (l,m,h)
                       search=None,       # Override search set on chat initialization (l,m,h)
                       stream=False,      # Stream results
                       max_steps=2, # Maximum number of tool calls
                       final_prompt=_final_prompt, # Final prompt when tool calls have ran out 
                       return_all=False,  # Returns all intermediate ModelResponses if not streaming and has tool calls
                       **kwargs):
        result_gen = self._call(msg, prefill, temp, think, search, stream, max_steps, 1, final_prompt, **kwargs)
        if stream or return_all: return result_gen
        async for res in result_gen: pass
        return res # normal chat behavior only return last msg

# %% ../nbs/00_core.ipynb
def _trunc_str(s, mx=2000, replace="<TRUNCATED>"):
    "Truncate `s` to `mx` chars max, adding `replace` if truncated"
    s = str(s).strip()
    return s[:mx]+replace if len(s)>mx else s

# %% ../nbs/00_core.ipynb
def mk_tr_details(tr, tc, mx=2000):
    "Create <details> block for tool call as JSON"
    args = {k:_trunc_str(v, mx=mx) for k,v in json.loads(tc.function.arguments).items()}
    res = {'id':tr['tool_call_id'], 
           'call':{'function': tc.function.name, 'arguments': args},
           'result':_trunc_str(tr.get('content'), mx=mx),}
    return f"\n\n{detls_tag}\n\n```json\n{dumps(res, indent=2)}\n```\n\n</details>\n\n"

# %% ../nbs/00_core.ipynb
class AsyncStreamFormatter:
    def __init__(self, include_usage=False, mx=2000):
        self.outp,self.tcs,self.include_usage,self.think,self.mx = '',{},include_usage,False,mx
    
    def format_item(self, o):
        "Format a single item from the response stream."
        res = ''
        if isinstance(o, ModelResponseStream):
            d = o.choices[0].delta
            if nested_idx(d, 'reasoning_content'): 
                self.think = True
                res += '🧠'
            elif self.think:
                self.think = False
                res += '\n\n'
            if c:=d.content: res+=c
        elif isinstance(o, ModelResponse):
            if self.include_usage: res += f"\nUsage: {o.usage}"
            if c:=getattr(o.choices[0].message,'tool_calls',None):
                self.tcs = {tc.id:tc for tc in c}
        elif isinstance(o, dict) and 'tool_call_id' in o:
            res += mk_tr_details(o, self.tcs.pop(o['tool_call_id']), mx=self.mx)
        self.outp+=res
        return res
    
    async def format_stream(self, rs):
        "Format the response stream for markdown display."
        async for o in rs: yield self.format_item(o)

# %% ../nbs/00_core.ipynb
async def adisplay_stream(rs):
    "Use IPython.display to markdown display the response stream."
    try: from IPython.display import display, Markdown
    except ModuleNotFoundError: raise ModuleNotFoundError("This function requires ipython. Please run `pip install ipython` to use.")
    fmt = AsyncStreamFormatter()
    md = ''
    async for o in fmt.format_stream(rs): 
        md+=o
        display(Markdown(md),clear=True)
    return fmt

## Enrich doc
"""Fix, clean markdown headings and enrich it with figures description, ..."""

# AUTOGENERATED! DO NOT EDIT! File to edit: ../nbs/04_enrichr.ipynb.

# %% auto 0
__all__ = ['ANTHROPIC_API_KEY', 'GEMINI_API_KEY', 'cfg', 'src_dir', 'log_dir', 'lm', 'setup_enhanced_dir', 'get_hdgs',
           'get_hdgs_with_pages', 'format_hdgs', 'HeadingResult', 'FixHeadingHierarchy', 'fix_md',
           'group_corrections_by_page', 'apply_corrections_to_page', 'apply_all_corrections', 'fix_doc_hdgs',
           'has_images', 'MarkdownPage', 'ImgRef', 'ImageRelevance', 'describe_img', 'copy_page_to_enriched',
           'process_single_page', 'enrich_images', 'md_plus_evaluation']

# %% ../nbs/04_enrichr.ipynb 3
from pathlib import Path
import os
import re
import json
import shutil
import time
from functools import partial
from tqdm import tqdm
from dotenv import load_dotenv
from fastcore.all import *
import dspy
from pydantic import BaseModel
from typing import List
from litellm import completion
import base64
from rich import print
import logging

# %% ../nbs/04_enrichr.ipynb 4
load_dotenv()
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

# %% ../nbs/04_enrichr.ipynb 5
cfg = AttrDict({
    'enhanced_dir': 'enhanced',
    'enriched_dir': 'enriched',
    'lm': 'gemini/gemini-2.0-flash',
    'api_key': GEMINI_API_KEY,
    'max_tokens': 8192,
    'track_usage': False,
    'img_dir': 'img'
})

# %% ../nbs/04_enrichr.ipynb 6
src_dir = Path("../_data/md_library/49d2fba781b6a7c0d94577479636ee6f")

# %% ../nbs/04_enrichr.ipynb 7
log_dir = Path.home() / '.evaluatr'
log_dir.mkdir(exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(str(log_dir / 'enrichr.log')),
        logging.StreamHandler()
    ]
)

# %% ../nbs/04_enrichr.ipynb 10
def setup_enhanced_dir(
    src_dir, # Source directory path
    enhanced_dir_name=cfg.enhanced_dir # Name of enhanced subdirectory
    ):
    "Create enhanced directory and copy all markdown files to it"
    src_path = Path(src_dir)
    enhanced_path = src_path / enhanced_dir_name
    enhanced_path.mkdir(exist_ok=True)
    for f in src_path.ls(file_exts=".md"): shutil.copy(f, enhanced_path)
    return enhanced_path

# %% ../nbs/04_enrichr.ipynb 13
def get_hdgs(md_txt): return re.findall(r'^#+.*$', md_txt, re.MULTILINE)

# %% ../nbs/04_enrichr.ipynb 14
def get_hdgs_with_pages(
    pages: list[Path] # List of pages
    ):
    "Get headings and the page number they are on"
    headings = []
    for i, page in enumerate(pages, 1):  # page numbers start at 1
        page_headings = get_hdgs(page.read_text())
        for o in page_headings: headings.append({'heading': o, 'page': i})
    return headings

# %% ../nbs/04_enrichr.ipynb 17
def format_hdgs(
    hdgs: list[dict] # List of headings with page numbers
    ):
    "Format headings with page numbers"
    formatted = []
    page_positions = {}
    
    for item in hdgs:
        page = item['page']
        page_positions[page] = page_positions.get(page, 0) + 1
        formatted.append(f"{item['heading']} (Page {page}, Position {page_positions[page]})")
    
    return "\n".join(formatted)

# %% ../nbs/04_enrichr.ipynb 19
lm = dspy.LM(cfg.lm, api_key=cfg.api_key)
dspy.configure(lm=lm)
dspy.settings.configure(track_usage=cfg.track_usage)

# %% ../nbs/04_enrichr.ipynb 20
class HeadingResult(BaseModel):
    old: str
    page: int
    position: int
    new: str
    changed: bool  # True if correction was made

# %% ../nbs/04_enrichr.ipynb 21
class FixHeadingHierarchy(dspy.Signature):
    """Fix markdown heading hierarchy by analyzing the document's numbering patterns:
    - Detect numbering scheme (1.2.3, I.A.1, A.1.a, etc.)
    - Apply hierarchy levels based on nesting depth: # for top level, ## for second level, ### for third level
    - When a section number is lower than a previously seen number at the same level (e.g., seeing '2.' after '3.1'), it's likely a subsection or list item, not a main section
    - Unnumbered headings: keep as-is if at document boundaries, treat as subsections if within numbered sections
    - Return ALL headings with their corrected form
    """
    
    headings_with_pages: str = dspy.InputField(desc="List of headings with page numbers")
    results: List[HeadingResult] = dspy.OutputField(desc="All headings with corrections and change status")

# %% ../nbs/04_enrichr.ipynb 22
def fix_md(
    hdgs: list[dict], # List of headings with page numbers
    track_usage: bool=cfg.track_usage,
    ):
    "Fix markdown headings"
    lm = dspy.LM(cfg.lm, api_key=cfg.api_key, max_tokens=cfg.max_tokens)
    dspy.configure(lm=lm)
    dspy.settings.configure(track_usage=track_usage)

    inp = format_hdgs(hdgs)
    fix_hdgs = dspy.ChainOfThought(FixHeadingHierarchy)
    result = fix_hdgs(headings_with_pages=inp)
    return result

# %% ../nbs/04_enrichr.ipynb 24
def group_corrections_by_page(
    results: list[HeadingResult], # List of headings with corrections and change status
    ):
    "Group HeadingResult corrections by page number into dict with page nums as keys"
    page_groups = {}
    for result in results:
        page = result.page
        if page not in page_groups:
            page_groups[page] = []
        page_groups[page].append(result)
    return page_groups

# %% ../nbs/04_enrichr.ipynb 26
def apply_corrections_to_page(
    page_nb, # Page number
    corrections, # List of corrections
    enhanced_path, # Path to enhanced directory
    ):
    "Apply corrections to a page in the enhanced directory"
    page_file = enhanced_path / f"page_{page_nb}.md"
    lines = page_file.read_text().splitlines()
    corrections_copy = corrections.copy()
    
    for i, line in enumerate(lines):
        for correction in corrections_copy:
            if line.strip() == correction.old.strip():
                lines[i] = f"{correction.new} .... page {page_nb}"
                corrections_copy.remove(correction)
                break
            
    page_file.write_text('\n'.join(lines))

# %% ../nbs/04_enrichr.ipynb 28
def apply_all_corrections(
    results, # List of headings with corrections and change status
    enhanced_path, # Path to enhanced directory
    ):
    "Apply all corrections to the pages in enhanced directory"
    grouped = group_corrections_by_page(results)
    for page_nb, corrections in grouped.items(): 
        apply_corrections_to_page(page_nb, corrections, enhanced_path)

# %% ../nbs/04_enrichr.ipynb 30
def fix_doc_hdgs(
    src_dir, # Path to the folder containing the document
    force=False, # Whether to overwrite the existing enhanced directory
    ):
    "Process the document directory"
    src_path = Path(src_dir)
    enhanced_path = src_path / cfg.enhanced_dir
    
    if enhanced_path.exists() and not force:
        print(f"Enhanced directory '{cfg.enhanced_dir}' already exists. Use force=True to overwrite.")
        return
    if enhanced_path.exists() and force: 
        shutil.rmtree(enhanced_path)
    
    enhanced_path = setup_enhanced_dir(src_dir)
    pages = enhanced_path.ls(file_exts=".md").sorted(key=lambda p: int(p.stem.split('_')[1]))
    result = fix_md(get_hdgs_with_pages(pages))
    apply_all_corrections(result.results, enhanced_path)

# %% ../nbs/04_enrichr.ipynb 34
def has_images(page_path):
    content = Path(page_path).read_text()
    return bool(re.search(r'!\[[^\]]*\]\([^)]+\)', content))

# %% ../nbs/04_enrichr.ipynb 37
class MarkdownPage: 
    "A class to represent a markdown page"
    def __init__(self, path): self.path = Path(path)

# %% ../nbs/04_enrichr.ipynb 38
class ImgRef(AttrDict):
    "A class to represent a image reference"
    def __repr__(self):
        clean_context = self.context.replace('\n', ' ')[:50] + "..."
        fields = [f"filename='{self.filename}'", f"context='{clean_context}'"]
        if hasattr(self, 'is_relevant'): fields.append(f"is_relevant={self.is_relevant}")
        if hasattr(self, 'reason'): fields.append(f"reason={self.reason}")
        # ... add other fields if present
        return f"ImgRef({', '.join(fields)})"


# %% ../nbs/04_enrichr.ipynb 39
@patch
def find_img_refs(
    self:MarkdownPage, # Markdown page of interest
    context_lines: int = 3, # Number of lines of context to include around the image
    ):
    "Find all image references in the markdown page and include the context around the image"
    content = self.path.read_text()
    lines = content.splitlines()
    results = []
    
    for i, line in enumerate(lines):
        if re.search(r'!\[[^\]]*\]\(([^)]+)\)', line):
            # Extract context around this line
            start = max(0, i - context_lines)
            end = min(len(lines), i + context_lines + 1)
            context = '\n'.join(lines[start:end])
            
            # Extract image filename
            match = re.search(r'!\[[^\]]*\]\(([^)]+)\)', line)
            results.append(ImgRef({
                "filename": match.group(1),
                "context": context
            }))
    
    return results

# %% ../nbs/04_enrichr.ipynb 42
class ImageRelevance(dspy.Signature):
    """Determine if an image contains substantive content for document understanding.
    
    RELEVANT: Charts, graphs, diagrams, figures, tables, screenshots, flowcharts
    IRRELEVANT: Logos, cover images, decorative elements, headers, footers
    """
    img_filename: str = dspy.InputField()
    surrounding_context: str = dspy.InputField(desc="Text context around the image")
    is_relevant: bool = dspy.OutputField(desc="True only for substantive content like data visualizations")
    reason: str = dspy.OutputField(desc="Brief explanation of decision")


# %% ../nbs/04_enrichr.ipynb 43
@patch
def classify_imgs(
    self:MarkdownPage, # Markdown page of interest
    img_refs: list[ImgRef], # List of image references
    ):
    "Classify images in the markdown page"
    classifier = dspy.ChainOfThought(ImageRelevance)
    for img_ref in img_refs:
        result = classifier(
            img_filename=img_ref.filename,
            surrounding_context=img_ref.context,
            page_nb=1  # We could make this dynamic if needed
        )
        img_ref.is_relevant = result.is_relevant
        img_ref.reason = result.reason
    return img_refs

# %% ../nbs/04_enrichr.ipynb 46
def describe_img(
    img_path: Path, # Path to the image
    context: str, # Context of the image
    api_key: str = cfg.api_key, # API key for the LLM model
    model: str = cfg.lm, # Model to use
    ):
    "Describe an image using an LLM"
    with open(img_path, "rb") as image_file:
        base64_image = base64.b64encode(image_file.read()).decode('utf-8')
    
    # Auto-detect image format
    img_format = img_path.suffix.lower().replace('.', '')
    if img_format == 'jpg': img_format = 'jpeg'
    
    prompt = f"""Provide a concise paragraph description of this image for evaluation report analysis. Include: type of content, main topic, key data/statistics, trends, and takeaways. Write as flowing text, not numbered points. Context: {context}"""
    response = completion(
        model=model,
        messages=[{
            "role": "user", 
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/{img_format};base64,{base64_image}"}}
            ]
        }],
        api_key=api_key
    )
    return response.choices[0].message.content

# %% ../nbs/04_enrichr.ipynb 47
@patch
def describe_imgs(
    self:MarkdownPage, # Markdown page of interest
    img_refs: list[ImgRef], # List of image references
    img_dir: str # Image directory
    ):
    "Describe images in the markdown page"
    for img_ref in img_refs:
        if img_ref.is_relevant:
            img_path = Path(img_dir) / img_ref.filename
            description = describe_img(img_path, img_ref.context, GEMINI_API_KEY)
            img_ref.description = description
    return img_refs

# %% ../nbs/04_enrichr.ipynb 50
@patch
def replace_imgs_with_desc(
    self:MarkdownPage, # Markdown page of interest
    img_refs, # List of image references
    enriched_dir: str = cfg.enriched_dir, # Enriched directory
    ):
    "Replace images with their descriptions in the markdown page"
    enriched_path = self.path.parent.parent / enriched_dir
    enriched_path.mkdir(exist_ok=True)
    
    content = self.path.read_text()
    for img_ref in img_refs:
        if img_ref.is_relevant and hasattr(img_ref, 'description'):
            pattern = f'!\\[[^\\]]*\\]\\({re.escape(img_ref.filename)}\\)'
            content = re.sub(pattern, img_ref.description, content)
    
    enriched_file = enriched_path / self.path.name
    enriched_file.write_text(content)
    return enriched_file

# %% ../nbs/04_enrichr.ipynb 51
def copy_page_to_enriched(
    page, # Page to copy
    enriched_dir: str = cfg.enriched_dir, # Enriched directory
    ):
    "Copy a page to the enriched directory"
    enriched_path = page.parent.parent / enriched_dir
    enriched_path.mkdir(exist_ok=True)
    return shutil.copy(page, enriched_path)

# %% ../nbs/04_enrichr.ipynb 52
def process_single_page(
    page, # Page to process
    img_dir, # Image directory
    enriched_dir: str = cfg.enriched_dir, # Enriched directory
    ):
    "Process a single page"
    md_page = MarkdownPage(page)
    # Pipeline: find → classify → describe → replace
    img_refs = md_page.find_img_refs()
    
    if not img_refs: return copy_page_to_enriched(page, enriched_dir)
    
    classified_refs = md_page.classify_imgs(img_refs)
    time.sleep(0.5)
    described_refs = md_page.describe_imgs(classified_refs, img_dir)
    time.sleep(0.5)
    return md_page.replace_imgs_with_desc(described_refs)

# %% ../nbs/04_enrichr.ipynb 53
def enrich_images(
    pages_dir, # Pages directory
    img_dir, # Image directory
    n_workers=2, # Number of workers
    ):
    "Enrich images in the pages directory"
    pages = Path(pages_dir).ls(file_exts=".md")
    
    pages_with_imgs = []
    for page in pages:
        if has_images(page):
            pages_with_imgs.append(page)
        else:
            copy_page_to_enriched(page)
    
    if pages_with_imgs:
        process_fn = partial(process_single_page, img_dir=img_dir)
        parallel(process_fn, pages_with_imgs, n_workers=n_workers, threadpool=True, progress=True)
        
    print(f"✓ Processed {len(pages)} pages ({len(pages_with_imgs)} with images)")

# %% ../nbs/04_enrichr.ipynb 56
@call_parse
def md_plus_evaluation(
    eval_id: str,  # Evaluation ID to process
    md_dir: str = "../data/md_library",  # Directory containing markdown folders
    overwrite: bool = False  # Overwrite if enhanced/enriched already exists
):
    "Fix markdown headings and enrich images for an evaluation report"
    # Find the evaluation's markdown directory
    eval_path = Path(md_dir) / eval_id
    
    if not eval_path.exists():
        logging.error(f"Markdown directory not found: {eval_path}")
        return
    
    # Find the actual report folder (first subdirectory)
    report_dirs = [d for d in eval_path.iterdir() if d.is_dir() 
                   and d.name != cfg.enhanced_dir and d.name != cfg.enriched_dir]
    
    if not report_dirs:
        logging.error(f"No report directory found in {eval_path}")
        return
    
    doc_path = report_dirs[0]
    
    # Check if already processed
    enhanced_path = doc_path / cfg.enhanced_dir
    enriched_path = doc_path.parent / cfg.enriched_dir
    
    if (enhanced_path.exists() or enriched_path.exists()) and not overwrite:
        logging.info(f"Output already exists for {eval_id}. Use --overwrite to reprocess.")
        return
    
    # Clean up if overwriting
    if overwrite:
        if enhanced_path.exists():
            shutil.rmtree(enhanced_path)
        if enriched_path.exists():
            shutil.rmtree(enriched_path)
    
    logging.info(f"Processing {eval_id}: fixing headings...")
    fix_doc_hdgs(doc_path, force=overwrite)
    
    logging.info(f"Processing {eval_id}: enriching images...")
    pages_dir = doc_path / cfg.enhanced_dir
    img_dir = doc_path / cfg.img_dir
    enrich_images(pages_dir, img_dir, n_workers=1)
    
    logging.info(f"Completed processing {eval_id}")

In [ ]:
mappr = nb_to_md("_06_mappr.ipynb")
mappr[:200]

'# Mappr\n\n> Scale up evaluation report mapping against evaluation frameworks using agentic workflows\n\n\n---\n\n::: {.callout-warning}\nThis notebook is a work in progress.\n:::\n\n---\n\nManually mapping evalua'

## Fix markdown headings


In [ ]:
#| export
def get_hdgs(
    md:str # Markdown file string
    ):
    "Return the markdown headings"
    # Sanitize removing '#' in python snippet if any
    md = sub(r'```[\s\S]*?```', '', md)
    return L(findall(r'^#{1,6} .+$', md, MULTILINE))



In [ ]:
#| eval: false
md = read_pgs('files/test/md_all/resnet')
hdgs = get_hdgs(md)
hdgs

(#22) ['# Deep Residual Learning for Image Recognition ','#### Abstract','## 1. Introduction','## 2. Related Work','## 3. Deep Residual Learning','### 3.1. Residual Learning','### 3.2. Identity Mapping by Shortcuts','### 3.3. Network Architectures','### 3.4. Implementation','## 4. Experiments','### 4.1. ImageNet Classification','### 4.2. CIFAR-10 and Analysis','### 4.3. Object Detection on PASCAL and MS COCO','## References','## A. Object Detection Baselines','## PASCAL VOC','## MS COCO','## B. Object Detection Improvements','## MS COCO','## PASCAL VOC'...]

In [ ]:
#| export
def fmt_hdgs_idx(
    hdgs: list[str] # List of markdown headings
    ) -> str: # Formatted string with index
    "Format the headings with index"
    return '\n'.join(f"{i}. {h}" for i, h in enumerate(hdgs))


In [ ]:
#| eval: false
hdgs_fmt = fmt_hdgs_idx(hdgs)
print(hdgs_fmt)

0. # Deep Residual Learning for Image Recognition 
1. #### Abstract
2. ## 1. Introduction
3. ## 2. Related Work
4. ## 3. Deep Residual Learning
5. ### 3.1. Residual Learning
6. ### 3.2. Identity Mapping by Shortcuts
7. ### 3.3. Network Architectures
8. ### 3.4. Implementation
9. ## 4. Experiments
10. ### 4.1. ImageNet Classification
11. ### 4.2. CIFAR-10 and Analysis
12. ### 4.3. Object Detection on PASCAL and MS COCO
13. ## References
14. ## A. Object Detection Baselines
15. ## PASCAL VOC
16. ## MS COCO
17. ## B. Object Detection Improvements
18. ## MS COCO
19. ## PASCAL VOC
20. ## ImageNet Detection
21. ## C. ImageNet Localization


In [ ]:
#| export
class HeadingCorrections(BaseModel):
    corrections: dict[int, str]  # index → corrected heading

In [ ]:
#| export
prompt_fix_hdgs = """Fix markdown heading hierarchy errors while preserving the document's intended structure.

INPUT FORMAT: Each heading is prefixed with its index number (e.g., "0. # Title")

RULES - Only fix these errors:
1. **Level jumps**: Headings can only increase by one # at a time
   - Wrong: 0. # Title → 1. #### Abstract
   - Fixed: 0. # Title → 1. ## Abstract

2. **Numbering inconsistency**: Subsection numbers must be one level deeper
   - Wrong: 4. ## 3. Section → 5. ## 3.1 Subsection
   - Fixed: 4. ## 3. Section → 5. ### 3.1 Subsection

3. **Preserve working structure**: If sections are consistently marked, keep it

4. **Decreasing levels is OK**: Going from ### to ## is valid for new sections

OUTPUT: Return a Python dictionary mapping index to corrected heading (without the index prefix).
Only include entries that need changes. Example: {{1: '## Abstract', 15: '### PASCAL VOC'}}

Headings to analyze:
{headings_list}
"""

In [ ]:
#| export
def fix_hdg_hierarchy(
    hdgs: list[str], # List of markdown headings
    model: str='claude-sonnet-4-5', # Model to use
    api_key: str=os.getenv('ANTHROPIC_API_KEY') # API key
    ) -> dict[int, str]: # Dictionary of index → corrected heading
    "Fix the heading hierarchy"
    r = completion(
        model=model, 
        messages=[{"role": "user", "content": prompt_fix_hdgs.format(headings_list=fmt_hdgs_idx(hdgs))}], 
        response_format=HeadingCorrections, 
        api_key=api_key
        )
    return json.loads(r.choices[0].message.content)['corrections']

In [ ]:
#| eval: false
fixes = fix_hdg_hierarchy(hdgs)
fixes

{'1': '## Abstract',
 '15': '### PASCAL VOC',
 '16': '### MS COCO',
 '18': '### MS COCO',
 '19': '### PASCAL VOC',
 '20': '### ImageNet Detection',
 '21': '### C. ImageNet Localization'}

In [ ]:
#| export
def mk_fixes_lut(
    hdgs: list[str], # List of markdown headings
    model: str='claude-sonnet-4-5', # Model to use
    api_key: str=os.getenv('ANTHROPIC_API_KEY') # API key
    ) -> dict[str, str]: # Dictionary of old → new heading
    "Make a lookup table of fixes"
    fixes = fix_hdg_hierarchy(hdgs, model, api_key)
    return {hdgs[int(k)]:v for k,v in fixes.items()}

In [ ]:
#| eval: false
lut_fixes = mk_fixes_lut(hdgs)
lut_fixes

{'#### Abstract': '## Abstract',
 '## PASCAL VOC': '### PASCAL VOC',
 '## MS COCO': '### MS COCO',
 '## ImageNet Detection': '### ImageNet Detection',
 '## C. ImageNet Localization': '### C. ImageNet Localization'}

In [ ]:
#| export
def apply_hdg_fixes(
    p:str, # Page to fix
    lut_fixes: dict[str, str], # Lookup table of fixes
    pg: int=None, # Optionnaly specify the page number to append to original heading
    ) -> str: # Page with fixes applied
    "Apply the fixes to the page"
    for old in get_hdgs(p): p = p.replace(old, lut_fixes.get(old, old) + (f' .... page {pg}' if pg else ''))
    return p

In [ ]:
#| eval: false
pg_nb = 1
p = read_pgs('files/test/md_all/resnet', join=False)[pg_nb-1]
print(apply_hdg_fixes(p, lut_fixes, pg=pg_nb)[:300])

# Deep Residual Learning for Image Recognition  .... page 1

Kaiming He Xiangyu Zhang Shaoqing Ren Jian Sun<br>Microsoft Research<br>\{kahe, v-xiangz, v-shren, jiansun\}@microsoft.com


## Abstract .... page 1

Deeper neural networks are more difficult to train. We present a residual learning framew
